This jupyter notebook need data source of PADUS4_1VectorAnalysis_PADUS_Only.gdb and import it to the current "utahdaminundationprofiles_aug9_2025" postgis databases.

In [1]:
import geopandas as gpd
import pandas as pd

In [3]:
gdb_file_path = r"/Users/xiaoliu/Work/Project/I-GUIDE/IGUIDE_Aging_Dam-selected/PADUS4_1VectorAnalysis_PADUS_Only.gdb"
layer_name = "PADUS4_1VectorAnalysis_PADUS_Only_Simp_SingP"  # Replace with the actual layer name

# Read the specific layer into a GeoDataFrame
try:
    gdf = gpd.read_file(gdb_file_path, layer=layer_name)
    print(f"Successfully loaded layer: {layer_name}")
except Exception as e:
    print(f"Error loading GDB layer: {e}")

/opt/homebrew/Caskroom/miniforge/base/envs/postgis-env/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


Successfully loaded layer: PADUS4_1VectorAnalysis_PADUS_Only_Simp_SingP


In [11]:
gdf.columns

Index(['FID_VectorAnalysisSelfUni1', 'FID_GAP_Sts14_13_12_12_11', 'Category',
       'Mang_Type', 'Mang_Name', 'Des_Tp', 'Loc_Ds', 'Unit_Nm', 'State_Nm',
       'Agg_Src', 'Pub_Access', 'GAP_Sts', 'IUCN_Cat', 'FeatClass',
       'GAP_Sts_Prity', 'COUNT_OBJECTID', 'ShL_ShA', 'DupShL_ShA', 'ORIG_FID',
       'RevOID', 'Shp_AreaDup', 'GIS_Acres', 'MngTp_Desc', 'MngTp_Reclass',
       'MngNm_Desc', 'DesTp_Desc', 'BndryName', 'BndryExten',
       'IUCN_Cat_Reclass', 'BndryID', 'GIS_AcrsDb', 'InPoly_FID', 'SimPgnFlag',
       'MaxSimpTol', 'MinSimpTol', 'Shape_Length', 'Shape_Area', 'geometry'],
      dtype='object')

In [21]:
gdf['GAP_Sts'] = gdf['GAP_Sts'].astype('int16')

In [22]:
status1and2_gdf = gdf[gdf['GAP_Sts'] <= 2]

In [25]:
status1and2_gdf['GAP_Sts'].head()

34378    2
34379    2
34380    2
34381    2
34382    2
Name: GAP_Sts, dtype: int16

In [26]:
columns_rename = ['FID_VectorAnalysisSelfUni1', 'FID_GAP_Sts14_13_12_12_11', 'Category',
       'Mang_Type', 'Mang_Name', 'Des_Tp', 'Loc_Ds', 'Unit_Nm', 'State_Nm',
       'Agg_Src', 'Pub_Access', 'gap_sts', 'IUCN_Cat', 'featclass',
       'GAP_Sts_Prity', 'COUNT_OBJECTID', 'ShL_ShA', 'DupShL_ShA', 'ORIG_FID',
       'RevOID', 'Shp_AreaDup', 'GIS_Acres', 'MngTp_Desc', 'MngTp_Reclass',
       'MngNm_Desc', 'DesTp_Desc', 'BndryName', 'BndryExten',
       'IUCN_Cat_Reclass', 'BndryID', 'GIS_AcrsDb', 'InPoly_FID', 'SimPgnFlag',
       'MaxSimpTol', 'MinSimpTol', 'Shape_Length', 'Shape_Area', 'geom']

In [27]:
status1and2_gdf.columns = columns_rename

In [31]:
status1and2_gdf = status1and2_gdf.set_geometry("geom")

In [32]:
target_crs = "EPSG:4326"
reprojected_gdf = status1and2_gdf.to_crs(target_crs)

print(f"Original CRS: {status1and2_gdf.crs}")
print(f"New CRS: {reprojected_gdf.crs}")

Original CRS: PROJCS["USA_Contiguous_Albers_Equal_Area_Conic_USGS_version",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","102039"]]
New CRS: EPSG:4326


In [33]:
import psycopg2


# ----------------------------
# Connect to PostGIS
# ----------------------------
conn = psycopg2.connect(
    dbname="utahdaminundationprofiles_aug9_2025",
    user="admin",
    password="admin",
    host="localhost",
    port=5432
)

In [34]:
from sqlalchemy import create_engine
import geopandas as gpd

# --- Assume these variables are derived from your psycopg2 connection details ---
# You must provide these credentials instead of the psycopg2 connection object itself
DB_USER = "admin"
DB_PASS = "admin"
DB_HOST = "localhost"
DB_PORT = "5432"  # Standard PostgreSQL port
DB_NAME = "utahdaminundationprofiles_aug9_2025"
table_name = "gap_status_1and2_padus4"

# Assuming 'merged_gdf' is your GeoDataFrame

# 1. CONSTRUCT THE POSTGRESQL CONNECTION URL
db_url = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# 2. CREATE THE SQLALCHEMY ENGINE
# GeoPandas requires a SQLAlchemy Engine object for to_postgis()
engine = create_engine(db_url)

# 3. UPLOAD THE GEODATAFRAME
try:
    reprojected_gdf.to_postgis(
        name=table_name,
        con=engine,          # Use the SQLAlchemy engine
        if_exists='replace', # or 'append'
        index=False 
    )
    print(f"Successfully uploaded GeoDataFrame to PostGIS table: {table_name}")
except Exception as e:
    print(f"Error uploading to PostGIS: {e}")

Successfully uploaded GeoDataFrame to PostGIS table: gap_status_1and2_padus4
